# Extraction Key Market Data

Ce notebook extrait les Key Market Data en fusionnant :
- **CSV S&P 500** : Base principale (~500 entreprises du S&P 500 avec Weight et Price)
- **CSV Performance** : Enrichissement avec métriques financières (Market Cap, Revenue, Net Income, etc.)
- **yfinance API** : Enrichissement avec secteur, industrie et **P/E Ratio** pour chaque entreprise
  - **P/E Ratio** : Price-to-Earnings Ratio = Prix de marché / Bénéfice par action (EPS)
  - Indique combien les investisseurs sont prêts à payer pour chaque dollar de bénéfices
  - Utilisé pour évaluer si une action est surévaluée ou sous-évaluée
- **Filtre fillings** : Garde uniquement les entreprises qui ont un 10-K dans `fillings/`

**Résultat** : `all_market_data.json` avec uniquement les entreprises S&P 500 qui ont un 10-K (mergées + yfinance)


## 1. Imports et Configuration


In [ ]:
import pandas as pd
import json
import time
from pathlib import Path
from typing import Dict, Optional
import yfinance as yf
from tqdm import tqdm

# Chemins
BASE_DIR = Path('../../')
DATA_RAW = BASE_DIR / 'data' / 'raw'
DATA_GENERATED = BASE_DIR / 'data' / 'generated' / 'key_market_data'

# Créer dossier de sortie
DATA_GENERATED.mkdir(parents=True, exist_ok=True)

print(f"📁 Dossier de sortie: {DATA_GENERATED}")


## 2. Charger CSV S&P 500 (Base principale - ~500 entreprises)


In [ ]:
# Charger CSV S&P 500 (contient Weight et Price pour les 503 tickers) - BASE PRINCIPALE
composition_df = pd.read_csv(DATA_RAW / '2025-08-15_composition_sp500.csv')

# Normaliser colonne Symbol -> Ticker
if 'Symbol' in composition_df.columns:
    composition_df = composition_df.rename(columns={'Symbol': 'Ticker'})

# Convertir Weight et Price (format européen avec virgule)
if 'Weight' in composition_df.columns:
    composition_df['Weight'] = composition_df['Weight'].astype(str).str.replace(',', '.').astype(float)
if 'Price' in composition_df.columns:
    composition_df['Price'] = composition_df['Price'].astype(str).str.replace(',', '.').astype(float)

print(f"✅ CSV S&P 500 chargé: {len(composition_df)} entreprises (BASE PRINCIPALE)")
print(f"   Colonnes: {list(composition_df.columns)}")
print(f"\n📊 Premières lignes:")
composition_df.head()


## 3. Charger CSV Performance (Enrichissement - métriques financières)


In [ ]:
# Charger CSV Performance (contient toutes les métriques financières) - ENRICHISSEMENT
performance_df = pd.read_csv(DATA_RAW / '2025-09-26_stocks-performance.csv')

# Normaliser colonne Symbol -> Ticker
if 'Symbol' in performance_df.columns:
    performance_df = performance_df.rename(columns={'Symbol': 'Ticker'})

print(f"✅ CSV Performance chargé: {len(performance_df)} entreprises (ENRICHISSEMENT)")
print(f"   Colonnes: {list(performance_df.columns)}")
print(f"\n📊 Premières lignes:")
performance_df.head()


## 4. Fusionner les données (S&P 500 BASE + Performance ENRICHISSEMENT)


In [ ]:
# Fusionner : S&P 500 comme BASE, enrichir avec Performance
# Méthode adaptée du collègue : merge par Ticker (left join sur S&P 500)
merged_df = pd.merge(
    composition_df,  # BASE PRINCIPALE : S&P 500 (~500 entreprises)
    performance_df[['Ticker', 'Company Name', 'Market Cap', 'Revenue', 'Op. Income', 'Net Income', 'EPS', 'FCF']],  # ENRICHISSEMENT : métriques financières
    on='Ticker',
    how='left',  # Garder tous les tickers du S&P 500, ajouter Performance si disponible
    suffixes=('', '_performance')
)

# FILTRER : Garder uniquement les entreprises qui ont un 10-K dans fillings/
fillings_dir = DATA_RAW / 'fillings'
available_tickers = [d.name for d in fillings_dir.iterdir() if d.is_dir()] if fillings_dir.exists() else []

print(f"\n📂 Filtrage par fillings disponibles:")
print(f"   Dossiers 10-K trouvés: {len(available_tickers)}")

# Filtrer merged_df pour garder seulement les tickers avec 10-K
merged_df = merged_df[merged_df['Ticker'].isin(available_tickers)].copy()

# Statistiques
total_count = len(merged_df)
performance_count = int(merged_df['Market Cap'].notna().sum())  # type: ignore

print(f"\n✅ Fusion et filtrage terminés: {total_count} entreprises (avec 10-K)")
print(f"   - Tickers avec données Performance: {performance_count}")
print(f"   - Tickers avec données S&P 500 seulement: {total_count - performance_count}")

merged_df.head()


## 5. Créer structure JSON pour chaque entreprise


In [ ]:
def create_market_data_structure(row: pd.Series) -> Dict:
    """
    Crée une structure JSON enrichie pour les données de marché d'une entreprise
    Les secteurs seront ajoutés plus tard via yfinance
    """
    # Helper function pour vérifier les valeurs None de manière sûre
    def _is_not_none(val):
        """Vérifie si une valeur n'est pas None/NaN de manière sûre pour le type checker"""
        if val is None:
            return False
        try:
            return pd.notna(val)  # type: ignore
        except (TypeError, ValueError):
            return False
    
    # Gérer company_name : priorité à Long Name (yfinance), puis Company (S&P 500), puis Company Name (performance)
    company_name = None
    long_name_val = row.get('Long Name')
    company_val = row.get('Company')
    company_name_val = row.get('Company Name')
    
    if _is_not_none(long_name_val):
        company_name = str(long_name_val).strip()
    elif _is_not_none(company_val):
        company_name = str(company_val).strip()
    elif _is_not_none(company_name_val):
        company_name = str(company_name_val).strip()
    
    # Toutes les entreprises sont dans S&P 500 (c'est notre base)
    is_sp500 = True  # Toujours True car on part du S&P 500
    
    # Structure de base
    ticker_val = row.get('Ticker')
    sector_val = row.get('Sector')
    industry_val = row.get('Industry')
    weight_val = row.get('Weight')
    price_val = row.get('Price')
    market_cap_val = row.get('Market Cap')
    revenue_val = row.get('Revenue')
    op_income_val = row.get('Op. Income')
    net_income_val = row.get('Net Income')
    eps_val = row.get('EPS')
    fcf_val = row.get('FCF')
    
    data = {
        "ticker": str(ticker_val).upper() if _is_not_none(ticker_val) else None,
        "company_name": company_name,
        "sector": str(sector_val) if _is_not_none(sector_val) else None,
        "industry": str(industry_val) if _is_not_none(industry_val) else None,
        "is_sp500": is_sp500,
        "sp500_weight": float(weight_val) if _is_not_none(weight_val) else None,  # type: ignore
        "current_price": float(price_val) if _is_not_none(price_val) else None,  # type: ignore
        "market_cap": float(market_cap_val) if _is_not_none(market_cap_val) else None,  # type: ignore
        "revenue": float(revenue_val) if _is_not_none(revenue_val) else None,  # type: ignore
        "operating_income": float(op_income_val) if _is_not_none(op_income_val) else None,  # type: ignore
        "net_income": float(net_income_val) if _is_not_none(net_income_val) else None,  # type: ignore
        "eps": float(eps_val) if _is_not_none(eps_val) else None,  # type: ignore
        "free_cash_flow": float(fcf_val) if _is_not_none(fcf_val) else None,  # type: ignore
    }
    
    # === CALCUL DES RATIOS FINANCIERS ===
    
    # 1. Profit Margin
    if data['revenue'] and data['net_income'] and data['revenue'] != 0:
        data['profit_margin'] = round(data['net_income'] / data['revenue'], 4)
    else:
        data['profit_margin'] = None
    
    # 2. Operating Margin
    if data['revenue'] and data['operating_income'] and data['revenue'] != 0:
        data['operating_margin'] = round(data['operating_income'] / data['revenue'], 4)
    else:
        data['operating_margin'] = None
    
    # 3. P/E Ratio (Price-to-Earnings Ratio)
    # Priorité: P/E depuis yfinance (données temps réel), sinon calcul depuis CSV
    # Le P/E Ratio indique combien les investisseurs sont prêts à payer pour chaque dollar de bénéfices
    # Utilisé pour évaluer si une action est surévaluée ou sous-évaluée
    # Calcul: Prix de marché / Bénéfice par action (EPS)
    
    # D'abord, utiliser le P/E depuis yfinance si disponible
    pe_yfinance_val = row.get('PE Ratio (yfinance)')
    if _is_not_none(pe_yfinance_val):
        data['pe_ratio'] = float(pe_yfinance_val)  # type: ignore
        data['pe_ratio_source'] = 'yfinance'  # Indiquer la source
    # Sinon, calculer depuis CSV (prix S&P 500 / EPS)
    elif data['current_price'] and data['eps'] and data['eps'] != 0:
        data['pe_ratio'] = round(data['current_price'] / data['eps'], 2)
        data['pe_ratio_source'] = 'calculated_csv'  # Indiquer la source
    else:
        data['pe_ratio'] = None
        data['pe_ratio_source'] = None
    
    # 4. Price to Sales
    if data['market_cap'] and data['revenue'] and data['revenue'] != 0:
        data['price_to_sales'] = round(data['market_cap'] / data['revenue'], 2)
    else:
        data['price_to_sales'] = None
    
    # 5. FCF Margin
    if data['revenue'] and data['free_cash_flow'] and data['revenue'] != 0:
        data['fcf_margin'] = round(data['free_cash_flow'] / data['revenue'], 4)
    else:
        data['fcf_margin'] = None
    
    # 6. FCF Yield
    if data['market_cap'] and data['free_cash_flow'] and data['market_cap'] != 0:
        data['fcf_yield'] = round(data['free_cash_flow'] / data['market_cap'], 4)
    else:
        data['fcf_yield'] = None
    
    # 7. Earnings Yield
    if data['current_price'] and data['eps'] and data['current_price'] != 0:
        data['earnings_yield'] = round(data['eps'] / data['current_price'], 4)
    else:
        data['earnings_yield'] = None
    
    # 8. Market Cap to Earnings
    if data['market_cap'] and data['net_income'] and data['net_income'] != 0:
        data['market_cap_to_earnings'] = round(data['market_cap'] / data['net_income'], 2)
    else:
        data['market_cap_to_earnings'] = None
    
    # 9. Operating Efficiency
    if data['operating_income'] and data['net_income'] and data['net_income'] != 0:
        data['operating_efficiency'] = round(data['operating_income'] / data['net_income'], 2)
    else:
        data['operating_efficiency'] = None
    
    # 10. Cash Conversion
    if data['net_income'] and data['free_cash_flow'] and data['net_income'] != 0:
        data['cash_conversion'] = round(data['free_cash_flow'] / data['net_income'], 2)
    else:
        data['cash_conversion'] = None
    
    return data

# Test sur une ligne
if len(merged_df) > 0:
    example = create_market_data_structure(merged_df.iloc[0])
    print("✅ Exemple de structure générée:")
    print(json.dumps(example, indent=2, ensure_ascii=False))


## 6. Enrichir avec yfinance (secteurs, industries et P/E Ratio)

**Cette étape enrichit les données AVANT de générer le JSON final.**

Récupère depuis l'API yfinance :
- **Secteur** et **Industrie** : Classification de l'entreprise
- **Long Name** : Nom complet officiel
- **P/E Ratio** : Price-to-Earnings Ratio (indique combien les investisseurs paient pour chaque dollar de bénéfices, utilisé pour évaluer si une action est surévaluée ou sous-évaluée)


In [ ]:
# Préparer les symboles pour Yahoo Finance
# Remplacer les virgules par des tirets (ex: BRK,B → BRK-B) pour yfinance
from typing import Optional

def normalize_ticker_for_yfinance(ticker: str) -> Optional[str]:
    """Normalise le ticker pour yfinance (remplace virgules par tirets)"""
    if pd.isna(ticker):  # type: ignore
        return None
    return str(ticker).replace(",", "-")

# Récupérer les secteurs, industries et P/E Ratio via yfinance
# Code adapté du collègue avec gestion d'erreurs et rate limiting
sector_dict = {}
industry_dict = {}
company_long_name_dict = {}
pe_ratio_dict = {}  # P/E Ratio depuis yfinance

print("🔍 Récupération des secteurs, industries et P/E Ratio via yfinance...")
print(f"   Total tickers à traiter: {len(merged_df)}")
print(f"   ⚠️ Cette étape peut prendre du temps (rate limiting API)\n")

# Limiter aux tickers uniques pour éviter les doublons
unique_tickers = merged_df['Ticker'].dropna().unique()  # type: ignore
print(f"   Tickers uniques: {len(unique_tickers)}")

# Option pour limiter le nombre de tickers (pour test rapide)
# LIMIT_TICKERS = 100  # Décommenter pour tester avec seulement 100 tickers
LIMIT_TICKERS = None  # None = tous les tickers

if LIMIT_TICKERS:
    unique_tickers = unique_tickers[:LIMIT_TICKERS]
    print(f"   ⚠️ Mode TEST: Limitation à {LIMIT_TICKERS} tickers")

for ticker in tqdm(unique_tickers, desc="Récupération yfinance"):
    try:
        # Normaliser le ticker pour yfinance
        yf_ticker = normalize_ticker_for_yfinance(ticker)
        if not yf_ticker:
            continue
            
        # Récupérer les infos via yfinance
        company = yf.Ticker(yf_ticker)
        info = company.info  # Dictionnaire avec toutes les infos
        
        # Extraire secteur, industrie et nom long
        sector_dict[ticker] = info.get("sector", "Unknown")
        industry_dict[ticker] = info.get("industry", "Unknown")
        company_long_name_dict[ticker] = info.get("longName", None)
        
        # P/E Ratio (Price-to-Earnings Ratio)
        # Indique combien les investisseurs sont prêts à payer pour chaque dollar de bénéfices
        # Utilisé pour évaluer si une action est surévaluée ou sous-évaluée
        trailing_pe = info.get("trailingPE")  # P/E basé sur les bénéfices passés
        forward_pe = info.get("forwardPE")  # P/E basé sur les bénéfices futurs estimés
        
        # Priorité à trailingPE, sinon forwardPE
        pe_ratio = trailing_pe or forward_pe
        
        # Si aucun P/E disponible, essayer de calculer depuis EPS et prix actuel
        if not pe_ratio:
            eps = info.get("trailingEps") or info.get("forwardEps")
            current_price = info.get("currentPrice") or info.get("regularMarketPrice")
            if eps and eps > 0 and current_price and current_price > 0:
                pe_ratio = current_price / eps
        
        pe_ratio_dict[ticker] = round(pe_ratio, 2) if pe_ratio else None
        
        # Délai pour éviter rate limiting (0.5 secondes entre chaque appel)
        time.sleep(0.5)
        
    except Exception as e:
        # Si erreur, mettre "Unknown" ou None
        sector_dict[ticker] = "Unknown"
        industry_dict[ticker] = "Unknown"
        company_long_name_dict[ticker] = None
        pe_ratio_dict[ticker] = None
        # print(f"   ⚠️ Erreur pour {ticker}: {str(e)}")

print("\n✅ Récupération terminée !")

# Ajouter les colonnes au DataFrame
merged_df['Sector'] = merged_df['Ticker'].map(sector_dict)  # type: ignore
merged_df['Industry'] = merged_df['Ticker'].map(industry_dict)  # type: ignore
merged_df['Long Name'] = merged_df['Ticker'].map(company_long_name_dict)  # type: ignore
merged_df['PE Ratio (yfinance)'] = merged_df['Ticker'].map(pe_ratio_dict)  # type: ignore

# Statistiques
print(f"\n📊 Statistiques des secteurs:")
sector_counts = merged_df['Sector'].value_counts()  # type: ignore
print(sector_counts.head(10))  # type: ignore

# Statistiques P/E Ratio
pe_available = int(merged_df['PE Ratio (yfinance)'].notna().sum())  # type: ignore
print(f"\n📊 Statistiques P/E Ratio:")
print(f"   Tickers avec P/E Ratio (yfinance): {pe_available}/{len(merged_df)}")
if pe_available > 0:
    pe_mean = float(merged_df['PE Ratio (yfinance)'].mean())  # type: ignore
    pe_median = float(merged_df['PE Ratio (yfinance)'].median())  # type: ignore
    print(f"   P/E Ratio moyen: {pe_mean:.2f}")
    print(f"   P/E Ratio médian: {pe_median:.2f}")

print(f"\n✅ Colonnes ajoutées: Sector, Industry, Long Name, PE Ratio (yfinance)")
print(f"\n📊 Aperçu:")
merged_df[['Ticker', 'Company', 'Sector', 'Industry', 'PE Ratio (yfinance)']].head(20)  # type: ignore


## 7. Générer le JSON final


In [ ]:
# Générer le JSON final avec TOUTES les données (mergées + yfinance)
# UN SEUL FICHIER JSON généré après tous les enrichissements
print("🔄 Génération du JSON final avec toutes les données...")

all_market_data_final = {}
for idx, row in merged_df.iterrows():
    ticker_val = row.get('Ticker')
    ticker = str(ticker_val).upper() if ticker_val is not None and pd.notna(ticker_val) else None  # type: ignore
    if ticker and ticker.strip():
        all_market_data_final[ticker] = create_market_data_structure(row)

# Sauvegarder le JSON final (SEUL fichier généré)
output_file = DATA_GENERATED / "all_market_data.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_market_data_final, f, indent=2, ensure_ascii=False)

print(f"\n✅ Fichier JSON final sauvegardé: {output_file}")
print(f"   Chemin absolu: {output_file.resolve()}")

# Statistiques finales
sector_count = sum(1 for v in all_market_data_final.values() if v.get('sector') and v.get('sector') != 'Unknown')
performance_count = sum(1 for v in all_market_data_final.values() if v.get('market_cap') is not None)

print(f"\n📊 Statistiques finales:")
print(f"   - Total entreprises (avec 10-K): {len(all_market_data_final)}")
print(f"   - Tickers avec secteur (yfinance): {sector_count}")
print(f"   - Tickers avec données Performance: {performance_count}")
print(f"   - Tickers avec prix S&P 500: {sum(1 for v in all_market_data_final.values() if v.get('current_price'))}")
print(f"\n✅ Fichier unique généré: all_market_data.json")


## 7. Exemples : Afficher quelques entreprises (avec secteurs)

In [ ]:
# Exemples S&P 500
print("📊 Exemple S&P 500 (AAPL):")
if 'AAPL' in all_market_data_final:
    print(json.dumps(all_market_data_final['AAPL'], indent=2, ensure_ascii=False))

print("\n" + "="*80 + "\n")

# Exemple d'une autre entreprise S&P 500
print("📊 Exemple S&P 500 (MSFT):")
if 'MSFT' in all_market_data_final:
    print(json.dumps(all_market_data_final['MSFT'], indent=2, ensure_ascii=False))

print("\n" + "="*80 + "\n")

# Résumé par secteur (comme dans le code du collègue)
print("📊 Résumé par secteur (Top 10):")
if 'Sector' in merged_df.columns:
    grouped = merged_df.groupby('Sector')
    for sector, df in list(grouped)[:10]:
        total_weight = df['Weight'].sum()
        print(f"\n=== {sector} ===")
        print(f"   Total entreprises S&P 500: {len(df)}")
        print(f"   Poids total dans S&P 500: {total_weight:.4f} ({total_weight*100:.2f}%)")
        print(f"   Exemples: {', '.join(df.head(5)['Ticker'].tolist())}")